# 01 · Closed-form n-th derivatives

**The one idea behind omnibias:** for the "Riccati class" of activations
(`sigmoid`, `tanh`, `softplus`, `gaussian`, `exp`, …) the *n*-th derivative
$\sigma^{(n)}(z)$ has a **closed form** — a polynomial in a *single* activation
evaluation, for **any** order $n$.

This notebook shows three things:

1. The closed form matches the analytic derivatives exactly.
2. It stays at **machine precision** where a finite-difference stencil falls
   apart (loses ~1 digit per order).
3. It costs **one** activation call at any order, while nested autograd grows.

Everything runs on CPU in a few seconds.

In [ ]:
import sys, time
import numpy as np
import torch
import matplotlib.pyplot as plt

sys.path.insert(0, ".")  # import the shared _style.py in notebooks/
from _style import set_style, ACCENT, GOOD, PRIMARY, INK, WARM
set_style()

from omnibias.torch.activations.registry import get_activation

torch.manual_seed(0)
print("torch", torch.__version__)

## 1. The closed form is exact

For `sigmoid`, with $s=\sigma(z)$, the Riccati identity $\sigma'=s(1-s)$ implies
a polynomial recursion $\sigma^{(n)} = P_n(s)$. We check the first few orders
against hand-written derivatives.

In [ ]:
z = torch.linspace(-3, 3, 7, dtype=torch.float64)
sig = get_activation("sigmoid")
s = torch.sigmoid(z)

# Hand-written low-order derivatives of the sigmoid.
manual = {
    0: s,
    1: s * (1 - s),
    2: s * (1 - s) * (1 - 2 * s),
    3: s * (1 - s) * (1 - 6 * s + 6 * s**2),
}
for n, ref in manual.items():
    got = sig.fastpath(z, n)
    err = (got - ref).abs().max().item()
    print(f"sigmoid^({n})  max|closed-form - manual| = {err:.2e}")

## 2. Accuracy: closed form vs a finite-difference stencil

The "obvious" way to get a high-order derivative numerically is to apply a
finite-difference operator repeatedly. Each application divides by the step
size, so round-off explodes with order. The closed form has no such division.

In [ ]:
tanh = get_activation("tanh")
orders = list(range(1, 9))
h = 5e-2
fd_err = []
for n in orders:
    half = n + 2
    xs = torch.linspace(-3, 3, 1201, dtype=torch.float64)
    pad = torch.arange(-half, half + 1, dtype=torch.float64) * h
    d = tanh.forward(xs[:, None] + pad[None, :])
    for _ in range(n):  # n-fold central difference
        d = (d[:, 2:] - d[:, :-2]) / (2 * h)
    center = d[:, d.shape[1] // 2]
    truth = tanh.fastpath(xs, n)
    fd_err.append(float(((center - truth).abs() / truth.abs().clamp_min(1e-3)).median()))

eps = float(np.finfo(np.float64).eps)
fig, ax = plt.subplots()
ax.semilogy(orders, fd_err, "o-", color=ACCENT, label="finite-difference stencil")
ax.semilogy(orders, [eps] * len(orders), "s-", color=GOOD, label="omnibias closed form")
ax.set_xlabel("derivative order  n"); ax.set_ylabel("median relative error")
ax.set_title("n-th derivative accuracy (tanh, float64)")
ax.legend(); plt.show()

## 3. Cost: one activation call at any order

Nested autograd builds a deeper graph for each extra order. The closed form
evaluates a polynomial in one activation call, so its cost is flat.

In [ ]:
z = torch.linspace(-3, 3, 20000, dtype=torch.float64)
orders = list(range(1, 7))

def t_fastpath(n):
    best = float("inf")
    for _ in range(5):
        t0 = time.perf_counter(); tanh.fastpath(z, n); best = min(best, time.perf_counter() - t0)
    return best * 1e3

def t_autograd(n):
    best = float("inf")
    for _ in range(3):
        zz = z.clone().requires_grad_(True)
        t0 = time.perf_counter()
        g = torch.autograd.grad(tanh.forward(zz).sum(), zz, create_graph=True)[0]
        for _ in range(n - 1):
            g = torch.autograd.grad(g.sum(), zz, create_graph=True)[0]
        best = min(best, time.perf_counter() - t0)
    return best * 1e3

fp = [t_fastpath(n) for n in orders]
ag = [t_autograd(n) for n in orders]
fig, ax = plt.subplots()
ax.plot(orders, ag, "o-", color=ACCENT, label="nested autograd")
ax.plot(orders, fp, "s-", color=GOOD, label="omnibias closed form")
ax.set_xlabel("derivative order  n"); ax.set_ylabel("time per call (ms)")
ax.set_title("Cost of the n-th derivative (20k points, CPU)")
ax.legend(); plt.show()

## Takeaway

Closed-form `σ⁽ⁿ⁾` is **exact**, **machine-precision stable**, and **flat in
cost** across order. That is the foundation everything else in omnibias builds
on: PINNs (notebook 03), Schrödinger solvers (04), VMC kinetic energy (05),
and closed-form curvature (06).

Next: **[02 · operator blocks](02_operator_blocks.ipynb)** — turning this into
typed layers.